# Создание многоклассовой модели XGBoost

# Загрузка

In [65]:
import pandas as pd
import numpy as np

In [66]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

In [67]:
import os
import csv
import joblib
from datetime import datetime

In [68]:
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.base import clone

# Первичный осмотр

In [69]:
df = pd.read_csv('input/heart_disease_uci.csv')
print(df.shape)
print(df['num'].value_counts().sort_index())

(920, 16)
num
0    411
1    265
2    109
3    107
4     28
Name: count, dtype: int64


# Очистка данных

In [70]:
df = df.drop(columns=['id'])
df.loc[df['chol'] == 0, 'chol'] = np.nan
df.loc[df['trestbps'] == 0, 'trestbps'] = np.nan
df.loc[df['oldpeak'] < 0, 'oldpeak'] = np.nan
# df = df.drop(columns=['dataset'])

# Разделение на X и y

In [71]:
X = df.drop(columns=['num'])
y = df['num']

# Train/validation/test split

In [72]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.2, stratify=y_train_val, random_state=42
)

# Препроцессинг через ColumnTransformer

In [88]:
def to_string(X):
    """Приводит категориальные признаки к строке."""
    return X.astype(str)

In [89]:
numeric_features = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca']
categorical_features = ['gender', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']

numeric_transformer = 'passthrough'

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('to_string', FunctionTransformer(to_string)),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [90]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [91]:
X_train_trans = preprocessor.fit_transform(X_train)
X_val_trans = preprocessor.transform(X_val)
X_test_trans = preprocessor.transform(X_test)

# Настройка XGBoost для многоклассовой классификации

In [92]:
model = XGBClassifier(
    objective='multi:softprob',
    num_class=5,
    eval_metric='mlogloss',
    tree_method='hist',
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50
)

# Учёт дисбаланса классов

In [93]:
sample_weight = compute_sample_weight(class_weight='balanced', y=y_train)

# Обучение с early stopping

In [94]:
model.fit(
    X_train_trans, y_train,
    sample_weight=sample_weight,
    eval_set=[(X_val_trans, y_val)],
    verbose=False
)

,"objective objective: str | xgboost.objective.Objective | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) re

# Оценка на тесте

In [95]:
y_pred = model.predict(X_test_trans)

In [96]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average='macro'))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.5869565217391305
Macro F1: 0.414821051566984
              precision    recall  f1-score   support

           0       0.87      0.76      0.81        82
           1       0.55      0.51      0.53        53
           2       0.28      0.32      0.30        22
           3       0.35      0.57      0.44        21
           4       0.00      0.00      0.00         6

    accuracy                           0.59       184
   macro avg       0.41      0.43      0.41       184
weighted avg       0.62      0.59      0.60       184

[[62 12  3  3  2]
 [ 9 27 10  6  1]
 [ 0  5  7  9  1]
 [ 0  4  4 12  1]
 [ 0  1  1  4  0]]


In [97]:
os.makedirs('results', exist_ok=True)

cm = confusion_matrix(y_test, y_pred)
report_dict = classification_report(y_test, y_pred, output_dict=True)

# Собираем строку для CSV
row = {
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'model': 'XGBClassifier',
    'accuracy': round(accuracy_score(y_test, y_pred), 4),
    'macro_f1': round(f1_score(y_test, y_pred, average='macro'), 4),
    'weighted_f1': round(f1_score(y_test, y_pred, average='weighted'), 4),
}

# Добавляем precision/recall/f1 по каждому классу
for cls in sorted(report_dict.keys()):
    if cls.isdigit():
        row[f'precision_class_{cls}'] = round(report_dict[cls]['precision'], 4)
        row[f'recall_class_{cls}']    = round(report_dict[cls]['recall'], 4)
        row[f'f1_class_{cls}']        = round(report_dict[cls]['f1-score'], 4)

file_path = 'results/xgboost_metrics.csv'
file_exists = os.path.isfile(file_path)

with open(file_path, 'a', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    if not file_exists:
        writer.writeheader()
    writer.writerow(row)

print(f"Метрики добавлены в {file_path}")

Метрики добавлены в results/xgboost_metrics.csv


# Подбор гиперпараметров для повышения accuracy

In [98]:
## Шаг 1. Обернуть препроцессинг и модель в Pipeline
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(
        objective='multi:softprob',
        num_class=5,
        eval_metric='mlogloss',
        tree_method='hist',
        random_state=42,
        n_jobs=1            # важно: n_jobs=1 внутри модели при n_jobs=-1 в поиске
    ))
])

In [99]:
## Шаг 2. Определить пространство поиска
param_dist = {
    'model__n_estimators':     [100, 200, 300, 500, 700],
    'model__max_depth':        [3, 4, 5, 6, 7, 9],
    'model__learning_rate':    [0.01, 0.03, 0.05, 0.1, 0.15],
    'model__subsample':        [0.6, 0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'model__gamma':            [0, 0.1, 0.3, 0.5],
    'model__reg_alpha':        [0, 0.1, 0.5, 1],
    'model__reg_lambda':       [0.5, 1, 1.5, 2, 3],
    'model__min_child_weight': [1, 3, 5]
}

In [100]:
## Шаг 3. Настроить CV
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = RandomizedSearchCV(
    pipe,
    param_distributions=param_dist,
    n_iter=50,                # начните с 30, если долго
    scoring='accuracy',       # для accuracy; для macro F1 — 'f1_macro'
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1,
    refit=True
)

In [101]:
## Шаг 4. Запустить поиск
search.fit(X_train, y_train)   # без sample_weight!

print("Best params:", search.best_params_)
print("Best CV accuracy:", search.best_score_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'model__subsample': 0.6, 'model__reg_lambda': 1.5, 'model__reg_alpha': 0.1, 'model__n_estimators': 500, 'model__min_child_weight': 3, 'model__max_depth': 4, 'model__learning_rate': 0.03, 'model__gamma': 0, 'model__colsample_bytree': 0.7}
Best CV accuracy: 0.5832101984644357


In [102]:
## Шаг 5. Оценить на тесте
y_pred = search.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:",      f1_score(y_test, y_pred, average='macro'))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Test Accuracy: 0.625
Macro F1: 0.4180970682532775
              precision    recall  f1-score   support

           0       0.81      0.84      0.83        82
           1       0.61      0.58      0.60        53
           2       0.36      0.36      0.36        22
           3       0.28      0.33      0.30        21
           4       0.00      0.00      0.00         6

    accuracy                           0.62       184
   macro avg       0.41      0.42      0.42       184
weighted avg       0.61      0.62      0.62       184

[[69  7  3  3  0]
 [11 31  7  4  0]
 [ 3  3  8  8  0]
 [ 2  9  2  7  1]
 [ 0  1  2  3  0]]


In [103]:
## Шаг 1. Сохранить лучшую модель из ноутбука
os.makedirs('models', exist_ok=True)
joblib.dump(search.best_estimator_, 'models/xgb_heart_accuracy.pkl')
print("Модель сохранена в models/xgb_heart_accuracy.pkl")

Модель сохранена в models/xgb_heart_accuracy.pkl


In [104]:
## Шаг 6. Обучение с early stopping
best_params = {k.replace('model__', ''): v for k, v in search.best_params_.items()}
best_params.update({
    'objective': 'multi:softprob',
    'num_class': 5,
    'eval_metric': 'merror',      # accuracy-подобная метрика
    'tree_method': 'hist',
    'random_state': 42,
    'n_jobs': -1,
    'early_stopping_rounds': 50
})

final_model = XGBClassifier(**best_params)
final_model.fit(
    X_train_trans, y_train,
    eval_set=[(X_val_trans, y_val)],
    verbose=False
)

y_pred_final = final_model.predict(X_test_trans)
print("Final Accuracy:", accuracy_score(y_test, y_pred_final))
print("Final Macro F1:", f1_score(y_test, y_pred_final, average='macro'))

Final Accuracy: 0.5978260869565217
Final Macro F1: 0.3314815384457638


In [105]:
## Шаг 7. Сравнить две ветки
os.makedirs('results', exist_ok=True)

report_dict = classification_report(y_test, y_pred, output_dict=True)

row = {
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'model': 'XGBClassifier_RandomSearch_accuracy',
    'accuracy':  round(accuracy_score(y_test, y_pred), 4),
    'macro_f1':  round(f1_score(y_test, y_pred, average='macro'), 4),
    'weighted_f1': round(f1_score(y_test, y_pred, average='weighted'), 4),
}
for cls in sorted(report_dict.keys()):
    if cls.isdigit():
        row[f'precision_class_{cls}'] = round(report_dict[cls]['precision'], 4)
        row[f'recall_class_{cls}']    = round(report_dict[cls]['recall'], 4)
        row[f'f1_class_{cls}']        = round(report_dict[cls]['f1-score'], 4)

file_path = 'results/xgboost_metrics.csv'
file_exists = os.path.isfile(file_path)

with open(file_path, 'a', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    if not file_exists:
        writer.writeheader()
    writer.writerow(row)

print(f"Ветка A записана в {file_path}")

Ветка A записана в results/xgboost_metrics.csv


In [106]:
## Отдельный search под macro F1
pipe_b = clone(pipe)   # свежий Pipeline, чтобы не тащить fitted состояние

cv_b = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search_b = RandomizedSearchCV(
    pipe_b,
    param_distributions=param_dist,
    n_iter=50,
    scoring='f1_macro',
    cv=cv_b,
    verbose=2,
    random_state=42,
    n_jobs=-1,
    refit=True
)

# веса считаем один раз на train и передаём через fit_params
sample_weight_train = compute_sample_weight(class_weight='balanced', y=y_train)

search_b.fit(X_train, y_train, model__sample_weight=sample_weight_train)

print("Best params (B):", search_b.best_params_)
print("Best CV macro F1:", search_b.best_score_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params (B): {'model__subsample': 0.9, 'model__reg_lambda': 0.5, 'model__reg_alpha': 0, 'model__n_estimators': 100, 'model__min_child_weight': 5, 'model__max_depth': 6, 'model__learning_rate': 0.03, 'model__gamma': 0.5, 'model__colsample_bytree': 0.6}
Best CV macro F1: 0.38800297447194526


In [107]:
## Оценка ветки B на тесте
y_pred_b = search_b.predict(X_test)

print("Branch B — Test Accuracy:", accuracy_score(y_test, y_pred_b))
print("Branch B — Macro F1:",      f1_score(y_test, y_pred_b, average='macro'))
print(classification_report(y_test, y_pred_b))
print(confusion_matrix(y_test, y_pred_b))

Branch B — Test Accuracy: 0.5597826086956522
Branch B — Macro F1: 0.41306913227287934
              precision    recall  f1-score   support

           0       0.85      0.76      0.80        82
           1       0.56      0.47      0.51        53
           2       0.23      0.23      0.23        22
           3       0.25      0.48      0.33        21
           4       0.25      0.17      0.20         6

    accuracy                           0.56       184
   macro avg       0.43      0.42      0.41       184
weighted avg       0.60      0.56      0.57       184

[[62 11  3  6  0]
 [ 9 25  9  9  1]
 [ 1  4  5 12  0]
 [ 1  5  3 10  2]
 [ 0  0  2  3  1]]


In [108]:
## Запись ветки B в CSV
report_dict_b = classification_report(y_test, y_pred_b, output_dict=True)

row_b = {
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'model': 'XGBClassifier_RandomSearch_f1macro',
    'accuracy':  round(accuracy_score(y_test, y_pred_b), 4),
    'macro_f1':  round(f1_score(y_test, y_pred_b, average='macro'), 4),
    'weighted_f1': round(f1_score(y_test, y_pred_b, average='weighted'), 4),
}
for cls in sorted(report_dict_b.keys()):
    if cls.isdigit():
        row_b[f'precision_class_{cls}'] = round(report_dict_b[cls]['precision'], 4)
        row_b[f'recall_class_{cls}']    = round(report_dict_b[cls]['recall'], 4)
        row_b[f'f1_class_{cls}']        = round(report_dict_b[cls]['f1-score'], 4)

with open(file_path, 'a', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=row_b.keys())
    writer.writerow(row_b)

print("Ветка B записана")

Ветка B записана


In [109]:
## Прочитать CSV и сравнить
df_metrics = pd.read_csv('results/xgboost_metrics.csv')
print(df_metrics[['timestamp', 'model', 'accuracy', 'macro_f1', 'weighted_f1']])

             timestamp                                model  accuracy  \
0  2026-10-01 21:59:20                        XGBClassifier    0.5870   
1  2026-10-01 21:59:47                        XGBClassifier    0.5870   
2  2026-10-01 22:00:51                        XGBClassifier    0.5870   
3  2026-10-01 22:21:42  XGBClassifier_RandomSearch_accuracy    0.6250   
4  2026-10-01 22:25:25   XGBClassifier_RandomSearch_f1macro    0.5598   
5  2026-10-01 22:34:51                        XGBClassifier    0.5870   
6  2026-10-01 22:45:01                        XGBClassifier    0.5870   
7  2026-10-01 22:46:20  XGBClassifier_RandomSearch_accuracy    0.6250   
8  2026-10-01 22:46:54   XGBClassifier_RandomSearch_f1macro    0.5598   

   macro_f1  weighted_f1  
0    0.4148       0.5991  
1    0.4148       0.5991  
2    0.4148       0.5991  
3    0.4181       0.6182  
4    0.4131       0.5746  
5    0.4148       0.5991  
6    0.4148       0.5991  
7    0.4181       0.6182  
8    0.4131       0.5746  